# Data Loading and Feature Engineering

This notebook loads the latest local OpenMementos feature build by default and optionally regenerates trace-level and block-level parquet partitions.

## Operations Performed

1. Configure the local feature-build location and choose either the latest existing build or a new run-specific output directory.
2. Optionally stream OpenMementos and write trace-level and block-level parquet partitions under `data/`.
3. Load the selected trace and block feature tables and normalize difficulty labels.
4. Create one grouped train/test split at the trace level, then propagate the same split to block rows.
5. Define block-level and trace-level high-compression targets using training-split thresholds only.
6. Save labeled block and trace feature tables for downstream modeling notebooks.
7. Run split-integrity and alignment checks so trace rows, block rows, labels, and sampled source rows remain auditable.

Full regeneration streams the Hugging Face dataset, parses each row into engineered trace-level and block-level feature tables, and saves local parquet files under `data/`. The `data/` directory is ignored by Git, so these generated files are not committed.

Keep `RUN_FULL_BUILD = False` unless deliberately rebuilding the local parquet files. The output is partitioned into multiple parquet files to avoid holding the full dataset in memory.

In [10]:
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

sys.path.append(str(Path("..").resolve()))

from src.reasoning_compression.alignment import get_original_rows_by_position
from src.reasoning_compression.features import (
    latest_feature_build_dir,
    normalize_difficulty,
    write_feature_partitions,
)

In [11]:
# --- feature build settings ---
# Normal runs should reuse an existing local build. Set this to True only when
# you intentionally want to stream OpenMementos and recreate the parquet files.
RUN_FULL_BUILD = False

# Location of timestamped local feature builds. This directory is ignored by Git.
FEATURE_BUILDS_DIR = Path("../data/full_feature_builds")

# Number of original traces to process before writing one parquet partition.
# Chunking keeps full-build memory usage bounded.
CHUNK_SIZE = 10_000

# Optional row limit for test builds. Use a small integer while debugging a full
# rebuild, then set to None when processing the complete dataset.
# MAX_ROWS = 1_000
MAX_ROWS = None

# --- persisted modeling split settings ---
MODEL_SPLIT_COLUMN = "model_split"
TRAIN_SPLIT_LABEL = "train"
TEST_SPLIT_LABEL = "test"
EXPECTED_SPLIT_LABELS = {TRAIN_SPLIT_LABEL, TEST_SPLIT_LABEL}
TEST_SIZE = 0.2
SPLIT_RANDOM_STATE = 42

build_configuration = {
    "run_full_build": RUN_FULL_BUILD,
    "feature_builds_dir": str(FEATURE_BUILDS_DIR),
    "chunk_size": CHUNK_SIZE,
    "max_rows": MAX_ROWS,
    "test_size": TEST_SIZE,
    "split_random_state": SPLIT_RANDOM_STATE,
}

build_configuration


{'run_full_build': False,
 'feature_builds_dir': '../data/full_feature_builds',
 'chunk_size': 10000,
 'max_rows': None,
 'test_size': 0.2,
 'split_random_state': 42}

In [12]:
# Choose the feature-build directory used by all following cells.
# A full build writes into a new timestamped directory. A normal run reads the
# latest completed local build.
if RUN_FULL_BUILD:
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    OUTPUT_DIR = FEATURE_BUILDS_DIR / run_id
else:
    OUTPUT_DIR = latest_feature_build_dir(FEATURE_BUILDS_DIR)

TRACE_DIR = OUTPUT_DIR / "traces"
BLOCK_DIR = OUTPUT_DIR / "blocks"

if RUN_FULL_BUILD:
    TRACE_DIR.mkdir(parents=True, exist_ok=True)
    BLOCK_DIR.mkdir(parents=True, exist_ok=True)

selected_build_paths = {
    "output_dir": str(OUTPUT_DIR),
    "trace_partitions": str(TRACE_DIR),
    "block_partitions": str(BLOCK_DIR),
}

selected_build_paths


{'output_dir': '../data/full_feature_builds/20260612_222038',
 'trace_partitions': '../data/full_feature_builds/20260612_222038/traces',
 'block_partitions': '../data/full_feature_builds/20260612_222038/blocks'}

In [13]:
# Safety check: this should usually be False.
# True means the next cell will stream the raw dataset and write new partitions.
RUN_FULL_BUILD


False

In [14]:
# Execute the expensive feature build only when necessary.
# Otherwise, record that this notebook is using the selected existing build.
if RUN_FULL_BUILD:
    build_summary = write_feature_partitions(
        output_dir=OUTPUT_DIR,
        chunk_size=CHUNK_SIZE,
        max_rows=MAX_ROWS,
    )
else:
    build_summary = {
        "output_dir": str(OUTPUT_DIR),
        "status": "using_existing_local_build",
        "trace_partitions": str(TRACE_DIR),
        "block_partitions": str(BLOCK_DIR),
    }

build_summary


{'output_dir': '../data/full_feature_builds/20260612_222038',
 'status': 'using_existing_local_build',
 'trace_partitions': '../data/full_feature_builds/20260612_222038/traces',
 'block_partitions': '../data/full_feature_builds/20260612_222038/blocks'}

## Load Selected Feature Tables

Read the selected local parquet partitions into the two DataFrames used by the rest of the notebook: `df_blocks_full` for block-level features and `df_traces_full` for trace-level features. The notebook checks their loaded shapes directly, rather than creating separate preview DataFrames, so the reported row counts describe the actual objects used downstream.

Difficulty labels are normalized here so missing values become an explicit `"not_applicable"` category. This avoids accidentally dropping non-code domains later when modeling code filters missing values.

In [15]:
df_blocks_full = pd.read_parquet(BLOCK_DIR)
df_traces_full = pd.read_parquet(TRACE_DIR)

if df_blocks_full.empty:
    raise ValueError(f"No block rows were loaded from {BLOCK_DIR}.")

if df_traces_full.empty:
    raise ValueError(f"No trace rows were loaded from {TRACE_DIR}.")

df_blocks_full["difficulty"] = df_blocks_full["difficulty"].map(
    normalize_difficulty
)
df_traces_full["difficulty"] = df_traces_full["difficulty"].map(
    normalize_difficulty
)

loaded_feature_shapes = {
    "trace_rows": len(df_traces_full),
    "trace_columns": len(df_traces_full.columns),
    "block_rows": len(df_blocks_full),
    "block_columns": len(df_blocks_full.columns),
}

loaded_feature_shapes


{'trace_rows': 228557,
 'trace_columns': 19,
 'block_rows': 2013510,
 'block_columns': 21}

## Create the Persistent Train/Test Split

Create one deterministic grouped split at the trace level, then copy that split to block rows. This keeps every block from the same reasoning trace in the same split and gives all downstream notebooks a shared held-out set.

The split is created before defining compression targets. If the target threshold were computed before splitting, the held-out test rows would influence the prediction target and make the evaluation less independent.

In [16]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=SPLIT_RANDOM_STATE,
)

train_trace_idx, test_trace_idx = next(
    splitter.split(
        df_traces_full,
        groups=df_traces_full["trace_id"],
    )
)

df_traces_full[MODEL_SPLIT_COLUMN] = TRAIN_SPLIT_LABEL
df_traces_full.loc[
    df_traces_full.index[test_trace_idx],
    MODEL_SPLIT_COLUMN,
] = TEST_SPLIT_LABEL

observed_trace_split_labels = set(
    df_traces_full[MODEL_SPLIT_COLUMN].dropna().unique()
)
if observed_trace_split_labels != EXPECTED_SPLIT_LABELS:
    raise ValueError(
        "Unexpected trace split labels: "
        f"expected {sorted(EXPECTED_SPLIT_LABELS)}, "
        f"got {sorted(observed_trace_split_labels)}."
    )

trace_split = df_traces_full[["trace_id", MODEL_SPLIT_COLUMN]]
df_blocks_full = df_blocks_full.merge(
    trace_split,
    on="trace_id",
    how="left",
    validate="many_to_one",
)

if df_blocks_full[MODEL_SPLIT_COLUMN].isna().any():
    raise ValueError("Some block rows do not have an assigned model split.")

split_summary = {
    "train_traces": int((df_traces_full[MODEL_SPLIT_COLUMN] == TRAIN_SPLIT_LABEL).sum()),
    "test_traces": int((df_traces_full[MODEL_SPLIT_COLUMN] == TEST_SPLIT_LABEL).sum()),
    "train_block_rows": int((df_blocks_full[MODEL_SPLIT_COLUMN] == TRAIN_SPLIT_LABEL).sum()),
    "test_block_rows": int((df_blocks_full[MODEL_SPLIT_COLUMN] == TEST_SPLIT_LABEL).sum()),
}

split_summary


{'train_traces': 182845,
 'test_traces': 45712,
 'train_block_rows': 1611167,
 'test_block_rows': 402343}

## Define the Block-Level Compression Target

Define the block-level high-compression label from the training split only. Lower `summary_to_block_token_ratio` values indicate stronger compression, so rows at or below the training-set 25th percentile are labeled as high compression.

The resulting `block_target_check` reports target prevalence by split before the block table is persisted. This makes it easier to spot unexpected class balance or threshold issues before downstream modeling notebooks reuse the labeled artifact.

In [17]:
block_compression_threshold = df_blocks_full.loc[
    df_blocks_full[MODEL_SPLIT_COLUMN] == TRAIN_SPLIT_LABEL,
    "summary_to_block_token_ratio",
].quantile(0.25)

if pd.isna(block_compression_threshold):
    raise ValueError(
        "Block compression threshold is missing; check training split rows."
    )

df_blocks_full["high_token_compression"] = (
    df_blocks_full["summary_to_block_token_ratio"] <= block_compression_threshold
).astype(int)

block_target_check = (
    df_blocks_full
    .groupby(MODEL_SPLIT_COLUMN)
    .agg(
        n_blocks=("high_token_compression", "size"),
        high_compression_share=("high_token_compression", "mean"),
    )
    .assign(block_compression_threshold=block_compression_threshold)
)

block_target_check.round(4)


,n_blocks,high_compression_share,block_compression_threshold
model_split,,,
test,402343,0.2497,0.1089
train,1611167,0.2500,0.1089


## Build Trace-Level Compression Features and Target

This section converts block-level compression behavior back into trace-level features. The block table has one row per `(reasoning block, summary)` pair, while the trace table has one row per original reasoning trace. To model trace-level behavior, the notebook first aggregates block rows by `trace_id` into `trace_compression`.

The aggregated summaries are then merged back into `df_traces_full` with a one-to-one validation on `trace_id`. This keeps the original trace metadata while adding compression features such as total block tokens, total summary tokens, mean and median block compression ratios, and the share of high-compression blocks.

The trace-level high-compression target belongs to the same conceptual step. After the aggregate features are attached, the notebook computes `trace_compression_threshold` from training traces only and labels `trace_high_token_compression` for both train and test traces. This mirrors the block-level target logic while keeping held-out traces out of threshold estimation.

In [18]:
# Create one compression summary row per original reasoning trace
trace_compression = (
    df_blocks_full
    .groupby("trace_id")
    .agg(
        trace_block_tokens=("block_tokens", "sum"),
        trace_summary_tokens=("summary_tokens", "sum"),
        trace_mean_summary_to_block_token_ratio=("summary_to_block_token_ratio", "mean"),
        trace_median_summary_to_block_token_ratio=("summary_to_block_token_ratio", "median"),
        trace_high_compression_share=("high_token_compression", "mean"),
    )
    .reset_index()
)

# The total trace compression ratio compares all summary tokens against all
# original block tokens within the same trace
trace_compression["trace_total_summary_to_block_token_ratio"] = (
    trace_compression["trace_summary_tokens"]
    / trace_compression["trace_block_tokens"]
)

trace_compression.head()

,trace_id,trace_block_tokens,trace_summary_tokens,trace_mean_summary_to_block_token_ratio,trace_median_summary_to_block_token_ratio,trace_high_compression_share,trace_total_summary_to_block_token_ratio
0,0,3267,898,0.297850,0.303896,0.00,0.274870
1,1,2110,712,0.335386,0.270494,0.00,0.337441
2,2,14102,982,0.132226,0.056338,0.75,0.069636
3,3,5021,733,0.205201,0.104587,0.60,0.145987
4,4,3305,1077,0.350994,0.274430,0.00,0.325870


In [19]:
# Merge trace-level compression summaries onto the original trace-level table.
df_traces_full = df_traces_full.merge(
    trace_compression,
    on="trace_id",
    how="left",
    validate="one_to_one",
)

trace_compression_columns = [
    "trace_block_tokens",
    "trace_summary_tokens",
    "trace_mean_summary_to_block_token_ratio",
    "trace_median_summary_to_block_token_ratio",
    "trace_high_compression_share",
    "trace_total_summary_to_block_token_ratio",
]

if df_traces_full[trace_compression_columns].isna().any().any():
    raise ValueError(
        "Some trace rows are missing aggregated compression summaries."
    )

df_traces_full.shape

(228557, 26)

In [20]:
# Define the trace-level high-compression target using the training split only.
trace_compression_threshold = df_traces_full.loc[
    df_traces_full[MODEL_SPLIT_COLUMN] == TRAIN_SPLIT_LABEL,
    "trace_total_summary_to_block_token_ratio",
].quantile(0.25)

if pd.isna(trace_compression_threshold):
    raise ValueError(
        "Trace compression threshold is missing; check training split rows."
    )

df_traces_full["trace_high_token_compression"] = (
    df_traces_full["trace_total_summary_to_block_token_ratio"]
    <= trace_compression_threshold
).astype(int)

trace_target_check = (
    df_traces_full
    .groupby(MODEL_SPLIT_COLUMN)
    .agg(
        n_traces=("trace_high_token_compression", "size"),
        high_compression_share=("trace_high_token_compression", "mean"),
    )
    .assign(trace_compression_threshold=trace_compression_threshold)
)

trace_target_check.round(4)

,n_traces,high_compression_share,trace_compression_threshold
model_split,,,
test,45712,0.2488,0.1327
train,182845,0.2500,0.1327


In [21]:
# Save the final labeled feature tables for downstream modeling notebooks.
# The block table includes model_split and high_token_compression.
# The trace table includes model_split, trace compression summaries, and trace_high_token_compression.
df_blocks_full.to_parquet(
    OUTPUT_DIR / "blocks_features_full_labeled.parquet",
    index=False,
)

df_traces_full.to_parquet(
    OUTPUT_DIR / "traces_features_full_labeled.parquet",
    index=False,
)


## Data Loading and Engineering Outputs

The full OpenMementos feature build produces two local labeled parquet files:

- `blocks_features_full_labeled.parquet`: block-level features, the persisted `model_split`, and the block-level `high_token_compression` target.
- `traces_features_full_labeled.parquet`: trace-level features, the persisted `model_split`, and the trace-level `trace_high_token_compression` target.

The `model_split` column is assigned once at the trace level and copied to block rows. High-compression thresholds are computed on the training split only, then applied to both training and test rows. This keeps the held-out test split out of target-threshold estimation.

The block-level table has one row per `(reasoning block, summary)` pair. The trace-level table has one row per original reasoning trace.

### Raw-to-Final Flow

1. Raw OpenMementos rows are streamed by `write_feature_partitions` when `RUN_FULL_BUILD = True`.
2. The streamed rows are written into two local parquet partition directories: `TRACE_DIR` for trace-level features and `BLOCK_DIR` for block-level features.
3. The notebook loads those directories into `df_traces_full` and `df_blocks_full`, then normalizes the `difficulty` column in both tables.
4. A grouped trace-level train/test split is added to `df_traces_full` and copied to `df_blocks_full` through `trace_split`.
5. `df_blocks_full` receives the block-level `high_token_compression` target using a threshold computed from training block rows only.
6. `df_blocks_full` is aggregated into `trace_compression`, which is merged back into `df_traces_full` as trace-level compression features.
7. `df_traces_full` receives the trace-level `trace_high_token_compression` target using a threshold computed from training trace rows only.
8. The final labeled artifacts are saved as `blocks_features_full_labeled.parquet` and `traces_features_full_labeled.parquet`.
9. The validation section checks split integrity, parquet read-back, and trace/block alignment.

### Main DataFrame Objects

- `df_traces_full`: trace-level table loaded from `TRACE_DIR`, then updated with normalized difficulty, `model_split`, trace compression summaries, and `trace_high_token_compression`. Its shape is checked immediately after loading.
- `df_blocks_full`: block-level table loaded from `BLOCK_DIR`, then updated with normalized difficulty, copied `model_split`, and `high_token_compression`. Its shape is checked immediately after loading.
- `trace_split`: two-column helper table used to copy each trace's split label onto its block rows.
- `trace_compression`: trace-level aggregation derived from `df_blocks_full` and merged back into `df_traces_full`.
- `block_target_check`, `trace_target_check`, `split_integrity_checks`, and `df_alignment_check`: audit objects used to inspect target prevalence, split validity, and trace/block alignment.

These files are saved under the ignored `data/` directory and are not committed to Git.

## Fixed Split Limitation

The modeling notebooks use one persisted grouped train/test split. This is a deliberate compromise.

The high-compression targets are defined from the training-split compression distribution and then applied to both training and test rows. Repeating the outer split or running grouped cross-validation would require recomputing the target threshold inside every split so that each test fold remains independent of its own target definition.

For this project, we accept the fixed split as a limitation because it keeps target construction leakage-free, preserves one common held-out set across prompt-level, block-level, and sequential notebooks, and keeps model comparisons easy to audit. The resulting metrics should be read as fixed-split estimates, not as split-uncertainty intervals. A future robustness notebook could add repeated grouped splits with fold-specific target thresholds.

# Data Alignment Checks

This section runs lightweight checks that the engineered trace-level and block-level tables, persisted split labels, and saved parquet artifacts are internally consistent. These checks are not feature construction steps, so they are grouped here after the output summary.

The first checks verify that the train/test split is non-empty, grouped by trace, present on both tables, and that the saved block-level and trace-level parquet artifacts can be read back from disk.

The local alignment checks then use the parquet tables already loaded in memory, so they do not stream the original dataset. During feature construction, `trace_id` is assigned according to streaming order and copied from trace rows to block rows. The checks verify that trace IDs are unique, block rows point to known traces, split labels match between trace and block tables, sampled block counts match trace-level counts, and sampled block indexes are contiguous.

Streamed original-row comparison is slow and kept as an explicit opt-in diagnostic. Leave `RUN_STREAMING_ALIGNMENT_CHECK = False` for normal notebook runs.

In [22]:
# Verify that the persisted split is non-empty, grouped, and complete
train_trace_ids = set(
    df_traces_full.loc[
        df_traces_full[MODEL_SPLIT_COLUMN] == TRAIN_SPLIT_LABEL,
        "trace_id",
    ]
)
test_trace_ids = set(
    df_traces_full.loc[
        df_traces_full[MODEL_SPLIT_COLUMN] == TEST_SPLIT_LABEL,
        "trace_id",
    ]
)

split_integrity_checks = {
    "n_train_traces": len(train_trace_ids),
    "n_test_traces": len(test_trace_ids),
    "n_trace_ids_in_both_splits": len(train_trace_ids.intersection(test_trace_ids)),
    "n_block_rows_without_split": int(df_blocks_full[MODEL_SPLIT_COLUMN].isna().sum()),
    "n_trace_rows_without_split": int(df_traces_full[MODEL_SPLIT_COLUMN].isna().sum()),
}

if split_integrity_checks["n_train_traces"] == 0:
    raise ValueError("The persisted model split has no training traces.")

if split_integrity_checks["n_test_traces"] == 0:
    raise ValueError("The persisted model split has no test traces.")

if split_integrity_checks["n_trace_ids_in_both_splits"] != 0:
    raise ValueError("Some trace IDs appear in both train and test splits.")

if split_integrity_checks["n_block_rows_without_split"] != 0:
    raise ValueError("Some block rows are missing a model split label.")

if split_integrity_checks["n_trace_rows_without_split"] != 0:
    raise ValueError("Some trace rows are missing a model split label.")

split_integrity_checks


{'n_train_traces': 182845,
 'n_test_traces': 45712,
 'n_trace_ids_in_both_splits': 0,
 'n_block_rows_without_split': 0,
 'n_trace_rows_without_split': 0}

In [23]:
# Smoke-check the saved block-level artifact can be read back from disk
pd.read_parquet(OUTPUT_DIR / "blocks_features_full_labeled.parquet").shape


(2013510, 23)

In [24]:
# Smoke-check the saved trace-level artifact can be read back from disk
pd.read_parquet(OUTPUT_DIR / "traces_features_full_labeled.parquet").shape


(228557, 27)

In [25]:
# Keep streamed source-row validation opt-in because it reads from the original dataset
RUN_STREAMING_ALIGNMENT_CHECK = False

# Use a small deterministic sample for local alignment inspection
trace_ids_to_check = (
    df_traces_full["trace_id"]
    .drop_duplicates()
    .sort_values()
    .head(3)
    .tolist()
)

if not trace_ids_to_check:
    raise ValueError("No trace IDs are available for alignment checks.")

trace_ids_to_check


[0, 1, 2]

In [26]:
# Compare trace IDs across tables before doing row-level sample checks
trace_table_ids = df_traces_full["trace_id"]
block_table_ids = df_blocks_full["trace_id"]
trace_ids_unique = trace_table_ids.unique()
block_trace_ids_unique = block_table_ids.unique()

local_alignment_summary = {
    "n_trace_rows": len(df_traces_full),
    "n_block_rows": len(df_blocks_full),
    "n_duplicate_trace_ids": int(trace_table_ids.duplicated().sum()),
    "n_block_trace_ids_missing_from_trace_table": len(
        set(block_trace_ids_unique).difference(trace_ids_unique)
    ),
    "n_trace_ids_without_block_rows": len(
        set(trace_ids_unique).difference(block_trace_ids_unique)
    ),
}

if local_alignment_summary["n_duplicate_trace_ids"] != 0:
    raise ValueError("Trace-level table contains duplicate trace IDs.")

if local_alignment_summary["n_block_trace_ids_missing_from_trace_table"] != 0:
    raise ValueError("Some block rows point to trace IDs absent from the trace table.")

# Confirm that copied block split labels still match their parent trace rows
trace_split_by_id = df_traces_full.set_index("trace_id")[MODEL_SPLIT_COLUMN]
local_alignment_summary["n_block_rows_with_split_mismatch"] = int(
    df_blocks_full[MODEL_SPLIT_COLUMN]
    .ne(block_table_ids.map(trace_split_by_id))
    .sum()
)

if local_alignment_summary["n_block_rows_with_split_mismatch"] != 0:
    raise ValueError("Some block rows have model_split values that differ from their trace rows.")

local_alignment_summary


{'n_trace_rows': 228557,
 'n_block_rows': 2013510,
 'n_duplicate_trace_ids': 0,
 'n_block_trace_ids_missing_from_trace_table': 0,
 'n_trace_ids_without_block_rows': 0,
 'n_block_rows_with_split_mismatch': 0}

In [27]:
# Build a compact trace/block comparison table for the sampled trace IDs
sample_trace_rows = df_traces_full.loc[
    df_traces_full["trace_id"].isin(trace_ids_to_check),
    [
        "trace_id",
        "domain",
        "source",
        "difficulty",
        "n_blocks",
        "n_summaries",
        MODEL_SPLIT_COLUMN,
    ],
].copy()

sample_block_rows = df_blocks_full.loc[
    df_blocks_full["trace_id"].isin(trace_ids_to_check)
]

# Summarize block rows so each sampled trace can be validated in one row
sample_block_summary = (
    sample_block_rows
    .groupby("trace_id")
    .agg(
        block_table_rows=("block_index", "size"),
        min_block_index=("block_index", "min"),
        max_block_index=("block_index", "max"),
        unique_block_indices=("block_index", "nunique"),
        block_split_labels=(MODEL_SPLIT_COLUMN, "nunique"),
        block_trace_n_blocks=("n_blocks_in_trace", "max"),
        block_trace_n_blocks_values=("n_blocks_in_trace", "nunique"),
    )
    .reset_index()
)

# Derive boolean checks for expected row counts, contiguous block indexes, and labels
df_alignment_check = (
    sample_trace_rows
    .merge(sample_block_summary, on="trace_id", how="left", validate="one_to_one")
    .fillna({
        "block_table_rows": 0,
        "unique_block_indices": 0,
        "block_split_labels": 0,
        "block_trace_n_blocks_values": 0,
    })
    .assign(
        expected_block_rows=lambda df: df[["n_blocks", "n_summaries"]].min(axis=1),
        block_rows_match_expected=lambda df: (
            df["block_table_rows"] == df["expected_block_rows"]
        ),
        block_trace_n_blocks_match=lambda df: (
            df["block_table_rows"].eq(0)
            | df["block_trace_n_blocks"].eq(df["n_blocks"])
        ),
        block_index_contiguous=lambda df: (
            df["block_table_rows"].eq(0)
            | (
                df["min_block_index"].eq(0)
                & df["max_block_index"].add(1).eq(df["unique_block_indices"])
                & df["unique_block_indices"].eq(df["block_table_rows"])
            )
        ),
        split_label_present=lambda df: df[MODEL_SPLIT_COLUMN].isin(EXPECTED_SPLIT_LABELS),
        one_block_trace_n_blocks_value=lambda df: (
            df["block_table_rows"].eq(0)
            | df["block_trace_n_blocks_values"].eq(1)
        ),
        one_block_split_label=lambda df: (
            df["block_table_rows"].eq(0)
            | df["block_split_labels"].eq(1)
        ),
    )
)

df_alignment_check


,trace_id,domain,source,difficulty,n_blocks,n_summaries,model_split,block_table_rows,min_block_index,max_block_index,...,block_split_labels,block_trace_n_blocks,block_trace_n_blocks_values,expected_block_rows,block_rows_match_expected,block_trace_n_blocks_match,block_index_contiguous,split_label_present,one_block_trace_n_blocks_value,one_block_split_label
0,0,code,stackexchange_codegolf,7.0,7,7,train,7,0,6,...,1,7,1,7,True,True,True,True,True,True
1,1,code,stackexchange_codegolf,7.0,8,8,train,8,0,7,...,1,8,1,8,True,True,True,True,True,True
2,2,code,stackexchange_codegolf,8.0,4,4,train,4,0,3,...,1,4,1,4,True,True,True,True,True,True


In [28]:
# Fail fast if any sampled trace violates the local alignment expectations
alignment_boolean_columns = [
    "block_rows_match_expected",
    "block_trace_n_blocks_match",
    "block_index_contiguous",
    "split_label_present",
    "one_block_trace_n_blocks_value",
    "one_block_split_label",
]

failed_alignment_rows = df_alignment_check.loc[
    ~df_alignment_check[alignment_boolean_columns].all(axis=1)
]

if not failed_alignment_rows.empty:
    raise ValueError(
        "Local alignment checks failed for trace IDs: "
        f"{failed_alignment_rows['trace_id'].tolist()}"
    )

df_alignment_check[["trace_id", *alignment_boolean_columns]]


,trace_id,block_rows_match_expected,block_trace_n_blocks_match,block_index_contiguous,split_label_present,one_block_trace_n_blocks_value,one_block_split_label
0,0,True,True,True,True,True,True
1,1,True,True,True,True,True,True
2,2,True,True,True,True,True,True


The optional streamed source-row lookup is imported from `src.reasoning_compression.alignment` to keep this notebook focused on data loading and validation.


In [29]:
# Retrieve original dataset rows only when the streamed alignment diagnostic is enabled
if RUN_STREAMING_ALIGNMENT_CHECK:
    original_rows = get_original_rows_by_position(trace_ids_to_check)
else:
    original_rows = {}

original_rows.keys()


dict_keys([])

In [30]:
# Compare sampled engineered metadata against freshly parsed source rows when enabled
if RUN_STREAMING_ALIGNMENT_CHECK:
    from src.reasoning_compression.features import parse_response

    streamed_alignment_rows = []

    for trace_id in trace_ids_to_check:
        original = original_rows[trace_id]
        parsed = parse_response(original["response"])
        trace_row = df_traces_full.query("trace_id == @trace_id").iloc[0]

        streamed_alignment_rows.append({
            "trace_id": trace_id,
            "original_domain": original["domain"],
            "engineered_domain": trace_row["domain"],
            "original_source": original["source"],
            "engineered_source": trace_row["source"],
            "original_difficulty": original["difficulty"],
            "engineered_difficulty": trace_row["difficulty"],
            "parsed_n_blocks": parsed["n_blocks"],
            "trace_n_blocks": trace_row["n_blocks"],
            "parsed_n_summaries": parsed["n_summaries"],
            "trace_n_summaries": trace_row["n_summaries"],
        })

    df_streamed_alignment_check = pd.DataFrame(streamed_alignment_rows)
else:
    df_streamed_alignment_check = pd.DataFrame({
        "status": ["skipped"],
        "reason": ["RUN_STREAMING_ALIGNMENT_CHECK is False"],
    })

df_streamed_alignment_check


,status,reason
0,skipped,RUN_STREAMING_ALIGNMENT_CHECK is False


In [31]:
# Pick one sampled trace for human inspection of the engineered trace row
TRACE_ID_INSPECT = trace_ids_to_check[0]

df_traces_full.query("trace_id == @TRACE_ID_INSPECT")


,trace_id,domain,source,difficulty,problem_chars,problem_tokens,problem_math_symbol_share,problem_has_multiple_choice,problem_has_code_fence,problem_question_mark_count,...,n_summaries,block_summary_delta,model_split,trace_block_tokens,trace_summary_tokens,trace_mean_summary_to_block_token_ratio,trace_median_summary_to_block_token_ratio,trace_high_compression_share,trace_total_summary_to_block_token_ratio,trace_high_token_compression
0,0,code,stackexchange_codegolf,7.0,931,227,0.01826,0,0,0,...,7,0,train,3267,898,0.29785,0.303896,0.0,0.27487,0


In [32]:
# Inspect the corresponding block rows and compression features for the same trace
df_blocks_full.query("trace_id == @TRACE_ID_INSPECT").sort_values("block_index")[[
    "trace_id",
    "block_index",
    "block_tokens",
    "summary_tokens",
    "summary_to_block_token_ratio",
    "token_compression_savings",
    "high_token_compression",
    MODEL_SPLIT_COLUMN,
]]


,trace_id,block_index,block_tokens,summary_tokens,summary_to_block_token_ratio,token_compression_savings,high_token_compression,model_split
0,0,0,200,77,0.385000,0.615000,0,train
1,0,1,737,111,0.150611,0.849389,0,train
2,0,2,654,133,0.203364,0.796636,0,train
3,0,3,342,92,0.269006,0.730994,0,train
4,0,4,385,117,0.303896,0.696104,0,train
5,0,5,598,234,0.391304,0.608696,0,train
6,0,6,351,134,0.381766,0.618234,0,train


In [33]:
# Optionally print source text beside parsed block and summary previews
if RUN_STREAMING_ALIGNMENT_CHECK:
    from src.reasoning_compression.features import parse_response

    original = original_rows[TRACE_ID_INSPECT]
    parsed = parse_response(original["response"])

    print("Problem preview:")
    print(original["problem"][:1000])

    print("\nFirst block preview:")
    print(parsed["blocks"][0][:1000])

    print("\nFirst summary preview:")
    print(parsed["summaries"][0][:1000])
else:
    print(
        "Original-text preview skipped. Set "
        "RUN_STREAMING_ALIGNMENT_CHECK = True to stream original rows."
    )


Original-text preview skipped. Set RUN_STREAMING_ALIGNMENT_CHECK = True to stream original rows.
